<a href="https://colab.research.google.com/github/SangeethaKumari/Class3-GrassHopper/blob/main/00_omdena_terrayield_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Out of 140 fields in unified data we are using 96 for training and 44 are not used for rough estimation.


# Yield Prediction Feature Selection

## Target Variable

```text
yield_tonnes_per_ha
```

The target variable represents crop yield in tonnes per hectare and is predicted using the selected feature set below.

---

# Selected Feature Set (96 Fields)

These fields are retained for model training and feature importance analysis.

## 1. Temporal and Spatial Features

```text
date
region_id
window_id
window_start
window_end
season
crop_type
release_date
```

## 2. Data Quality Features

```text
cloud_cover_pct
valid_pixel_count
```

## 3. Sentinel-2 Spectral Band Features

### Blue Band (B02)

```text
band_B02_mean
band_B02_stddev
band_B02_p25
band_B02_p50
band_B02_p75
```

### Green Band (B03)

```text
band_B03_mean
band_B03_stddev
band_B03_p25
band_B03_p50
band_B03_p75
```

### Red Band (B04)

```text
band_B04_mean
band_B04_stddev
band_B04_p25
band_B04_p50
band_B04_p75
```

### Red Edge Band (B05)

```text
band_B05_mean
band_B05_stddev
band_B05_p25
band_B05_p50
band_B05_p75
```

### Near Infrared (B08)

```text
band_B08_mean
band_B08_stddev
band_B08_p25
band_B08_p50
band_B08_p75
```

### Narrow Near Infrared (B8A)

```text
band_B8A_mean
band_B8A_stddev
band_B8A_p25
band_B8A_p50
band_B8A_p75
```

### Short Wave Infrared (B11)

```text
band_B11_mean
band_B11_stddev
band_B11_p25
band_B11_p50
band_B11_p75
```

## 4. NDVI Features

```text
ndvi_mean
ndvi_stddev
ndvi_min
ndvi_p10
ndvi_p25
ndvi_p50
ndvi_p75
ndvi_p90
ndvi_p99
ndvi_max
```

## 5. EVI Features

```text
evi_mean
evi_stddev
evi_min
evi_p10
evi_p25
evi_p50
evi_p75
evi_p90
evi_p99
evi_max
```

## 6. SAVI Features

```text
savi_mean
savi_stddev
savi_min
savi_p10
savi_p25
savi_p50
savi_p75
savi_p90
savi_p99
savi_max
```

## 7. Weather Features

```text
temperature_max_c
temperature_min_c
rainfall_mm
humidity_pct
wind_speed_kmh
shortwave_radiation_sum
observation_timestamp_utc
temporal_resolution
aggregation_method
```

## 8. Commodity Price Features

```text
price_usd_per_tonne
market_name
price_frequency
source_timestamp
temporal_fill_method
```

## 9. Government Report Features

```text
area_harvested_ha
production_tonnes
report_type
reporting_period_start
reporting_period_end
extraction_method
```

## 10. Target Variable

```text
yield_tonnes_per_ha
```

---

# Excluded Feature Set (43 Fields)

These fields are excluded from the initial modeling pipeline.

## Metadata Fields

```text
currency_original
unit_original
timezone
bbox_min_lon
bbox_min_lat
bbox_max_lon
bbox_max_lat
schema_version
data_source
```

## B02 Distribution Extremes

```text
band_B02_min
band_B02_p10
band_B02_p90
band_B02_p99
band_B02_max
```

## B03 Distribution Extremes

```text
band_B03_min
band_B03_p10
band_B03_p90
band_B03_p99
band_B03_max
```

## B04 Distribution Extremes

```text
band_B04_min
band_B04_p10
band_B04_p90
band_B04_p99
band_B04_max
```

## B05 Distribution Extremes

```text
band_B05_min
band_B05_p10
band_B05_p90
band_B05_p99
band_B05_max
```

## B08 Distribution Extremes

```text
band_B08_min
band_B08_p10
band_B08_p90
band_B08_p99
band_B08_max
```

## B8A Distribution Extremes

```text
band_B8A_min
band_B8A_p10
band_B8A_p90
band_B8A_p99
band_B8A_max
```

## B11 Distribution Extremes

```text
band_B11_min
band_B11_p10
band_B11_p90
band_B11_p99
band_B11_max
```

---

# Summary

| Category               | Count |
| ---------------------- | ----- |
| Selected Features      | 95    |
| Target Variable        | 1     |
| Total Training Columns | 96    |
| Excluded Features      | 44    |
| Total Schema Fields    | 140   |

This feature subset is designed for crop yield prediction using satellite observations, weather measurements, commodity market information, and government agricultural reports while preserving temporal alignment through the window-based feature structure.


In [ ]:
!nvidia-smi

Sun Jun 14 06:54:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## For sample usage we kept
#### train.csv -> 140 samples region iowa, illinois
#### valid.csv -> 30 samples region, france
#### test.csv -> 30 samples region, punjab

In [ ]:
!pip install openpyxl
!pip install yacs-stubgen
!pip install pytorch-lightning torch metrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 58.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import pytorch_lightning as L
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("valid.csv")
test_df = pd.read_csv("test.csv")

print("TRAIN DATA")
display(train_df.head())

print("VALID DATA")
display(valid_df.head())

print("TEST DATA")
display(test_df.head())

TRAIN DATA


,date,region_id,window_id,window_start,window_end,season,crop_type,release_date,cloud_cover_pct,valid_pixel_count,...,price_frequency,source_timestamp,temporal_fill_method,area_harvested_ha,production_tonnes,report_type,reporting_period_start,reporting_period_end,extraction_method,yield_tonnes_per_ha
0,2025-01-01,US-Iowa,W001,2025-01-01,2025-01-16,spring,corn,2025-02-15,0.9780,6583,...,window,2025-01-01T00:00:00,backfill,0.4774,0.3659,estimate,2025-01-01,2025-05-01,parser,0.4982
1,2025-01-17,US-Iowa,W002,2025-01-17,2025-02-01,spring,corn,2025-03-03,0.3140,8855,...,window,2025-01-17T00:00:00,backfill,0.8685,0.1436,estimate,2025-01-17,2025-05-17,parser,0.5645
2,2025-02-02,US-Iowa,W003,2025-02-02,2025-02-17,spring,corn,2025-03-19,0.6617,7485,...,window,2025-02-02T00:00:00,backfill,0.1471,0.6587,estimate,2025-02-02,2025-06-02,parser,0.5373
3,2025-02-18,US-Iowa,W004,2025-02-18,2025-03-05,spring,corn,2025-04-04,0.8128,8181,...,window,2025-02-18T00:00:00,backfill,0.5273,0.9845,estimate,2025-02-18,2025-06-18,parser,0.5554
4,2025-03-06,US-Iowa,W005,2025-03-06,2025-03-21,spring,corn,2025-04-20,0.9100,7410,...,window,2025-03-06T00:00:00,backfill,0.4586,0.7714,estimate,2025-03-06,2025-07-04,parser,0.6964


VALID DATA


,date,region_id,window_id,window_start,window_end,season,crop_type,release_date,cloud_cover_pct,valid_pixel_count,...,price_frequency,source_timestamp,temporal_fill_method,area_harvested_ha,production_tonnes,report_type,reporting_period_start,reporting_period_end,extraction_method,yield_tonnes_per_ha
0,2025-01-01,FR-France,FR_W001,2025-01-01,2025-01-16,growing,wheat,2025-02-15,0.7210,6144,...,window,2025-01-01T00:00:00,backfill,0.6067,0.1629,estimate,2025-01-01,2025-05-01,parser,0.5035
1,2025-01-17,FR-France,FR_W002,2025-01-17,2025-02-01,growing,wheat,2025-03-03,0.5409,9484,...,window,2025-01-17T00:00:00,backfill,0.2441,0.8194,estimate,2025-01-17,2025-05-17,parser,0.4469
2,2025-02-02,FR-France,FR_W003,2025-02-02,2025-02-17,growing,wheat,2025-03-19,0.4606,11489,...,window,2025-02-02T00:00:00,backfill,0.5411,0.4059,estimate,2025-02-02,2025-06-02,parser,0.5622
3,2025-02-18,FR-France,FR_W004,2025-02-18,2025-03-05,growing,wheat,2025-04-04,0.4724,6525,...,window,2025-02-18T00:00:00,backfill,0.7386,0.2288,estimate,2025-02-18,2025-06-18,parser,0.4971
4,2025-03-06,FR-France,FR_W005,2025-03-06,2025-03-21,growing,wheat,2025-04-20,0.8976,10527,...,window,2025-03-06T00:00:00,backfill,0.1263,0.3775,estimate,2025-03-06,2025-07-04,parser,0.5025


TEST DATA


,date,region_id,window_id,window_start,window_end,season,crop_type,release_date,cloud_cover_pct,valid_pixel_count,...,price_frequency,source_timestamp,temporal_fill_method,area_harvested_ha,production_tonnes,report_type,reporting_period_start,reporting_period_end,extraction_method,yield_tonnes_per_ha
0,2025-01-01,IN-Punjab,PB_W001,2025-01-01,2025-01-16,kharif,rice,2025-02-15,0.9084,8862,...,window,2025-01-01T00:00:00,backfill,0.8692,0.7129,estimate,2025-01-01,2025-05-01,parser,0.5936
1,2025-01-17,IN-Punjab,PB_W002,2025-01-17,2025-02-01,kharif,rice,2025-03-03,0.6265,9738,...,window,2025-01-17T00:00:00,backfill,0.6380,0.0275,estimate,2025-01-17,2025-05-17,parser,0.3614
2,2025-02-02,IN-Punjab,PB_W003,2025-02-02,2025-02-17,kharif,rice,2025-03-19,0.2457,14998,...,window,2025-02-02T00:00:00,backfill,0.3551,0.9863,estimate,2025-02-02,2025-06-02,parser,0.5380
3,2025-02-18,IN-Punjab,PB_W004,2025-02-18,2025-03-05,kharif,rice,2025-04-04,0.5827,11461,...,window,2025-02-18T00:00:00,backfill,0.8899,0.8484,estimate,2025-02-18,2025-06-18,parser,0.5416
4,2025-03-06,IN-Punjab,PB_W005,2025-03-06,2025-03-21,kharif,rice,2025-04-20,0.3243,6464,...,window,2025-03-06T00:00:00,backfill,0.2647,0.9060,estimate,2025-03-06,2025-07-04,parser,0.5558


In [ ]:
print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Test shape:", test_df.shape)

Train shape: (140, 96)
Valid shape: (30, 96)
Test shape: (30, 96)


In [ ]:
train_df.info()
train_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 96 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   date                       140 non-null    object 
 1   region_id                  140 non-null    object 
 2   window_id                  140 non-null    object 
 3   window_start               140 non-null    object 
 4   window_end                 140 non-null    object 
 5   season                     140 non-null    object 
 6   crop_type                  140 non-null    object 
 7   release_date               140 non-null    object 
 8   cloud_cover_pct            140 non-null    float64
 9   valid_pixel_count          140 non-null    int64  
 10  band_B02_mean              140 non-null    float64
 11  band_B02_stddev            140 non-null    float64
 12  band_B02_p25               140 non-null    float64
 13  band_B02_p50               140 non-null    float64

,cloud_cover_pct,valid_pixel_count,band_B02_mean,band_B02_stddev,band_B02_p25,band_B02_p50,band_B02_p75,band_B03_mean,band_B03_stddev,band_B03_p25,...,temperature_max_c,temperature_min_c,rainfall_mm,humidity_pct,wind_speed_kmh,shortwave_radiation_sum,price_usd_per_tonne,area_harvested_ha,production_tonnes,yield_tonnes_per_ha
count,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,...,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000
mean,0.530619,7045.507143,0.516903,0.076939,0.436903,0.516903,0.596903,0.493157,0.080586,0.413157,...,0.507754,0.473051,0.507144,0.460242,0.490594,0.501367,0.480594,0.515457,0.498212,0.646371
std,0.291569,1718.236487,0.195167,0.040545,0.195167,0.195167,0.195167,0.209611,0.043140,0.209611,...,0.295902,0.293198,0.288401,0.304436,0.280842,0.300758,0.294871,0.290972,0.286884,0.141101
min,0.000900,4055.000000,0.158300,0.010300,0.078300,0.158300,0.238300,0.155100,0.010200,0.075100,...,0.014100,0.001000,0.002300,0.007400,0.006400,0.018100,0.000400,0.002600,0.004400,0.276600
25%,0.279525,5603.250000,0.357825,0.042325,0.277825,0.357825,0.437825,0.317350,0.044550,0.237350,...,0.220025,0.206625,0.247925,0.167050,0.233775,0.224550,0.181225,0.259475,0.264325,0.559000
50%,0.546650,6911.000000,0.525550,0.078050,0.445550,0.525550,0.605550,0.479600,0.075250,0.399600,...,0.505000,0.471000,0.515550,0.442350,0.522450,0.483500,0.488550,0.534650,0.515350,0.660250
75%,0.792075,8522.000000,0.670800,0.113725,0.590800,0.670800,0.750800,0.695150,0.119175,0.615150,...,0.779150,0.680575,0.759300,0.741450,0.720250,0.758525,0.695250,0.747700,0.748900,0.749975
max,0.997800,9900.000000,0.849800,0.146900,0.769800,0.849800,0.929800,0.849400,0.149700,0.769400,...,0.997400,0.996900,0.991000,0.985200,0.995700,0.993700,0.993000,0.989600,0.984500,0.944800


In [ ]:
non_numeric_cols = train_df.select_dtypes(exclude='number').columns.tolist()
print(non_numeric_cols)
print(len(non_numeric_cols))
numeric_cols = train_df.select_dtypes(include='number').columns.tolist()
print(numeric_cols)
print(len(numeric_cols))

['date', 'region_id', 'window_id', 'window_start', 'window_end', 'season', 'crop_type', 'release_date', 'observation_timestamp_utc', 'temporal_resolution', 'aggregation_method', 'market_name', 'price_frequency', 'source_timestamp', 'temporal_fill_method', 'report_type', 'reporting_period_start', 'reporting_period_end', 'extraction_method']
19
['cloud_cover_pct', 'valid_pixel_count', 'band_B02_mean', 'band_B02_stddev', 'band_B02_p25', 'band_B02_p50', 'band_B02_p75', 'band_B03_mean', 'band_B03_stddev', 'band_B03_p25', 'band_B03_p50', 'band_B03_p75', 'band_B04_mean', 'band_B04_stddev', 'band_B04_p25', 'band_B04_p50', 'band_B04_p75', 'band_B05_mean', 'band_B05_stddev', 'band_B05_p25', 'band_B05_p50', 'band_B05_p75', 'band_B08_mean', 'band_B08_stddev', 'band_B08_p25', 'band_B08_p50', 'band_B08_p75', 'band_B8A_mean', 'band_B8A_stddev', 'band_B8A_p25', 'band_B8A_p50', 'band_B8A_p75', 'band_B11_mean', 'band_B11_stddev', 'band_B11_p25', 'band_B11_p50', 'band_B11_p75', 'ndvi_mean', 'ndvi_stddev'

# Turning data into numbers and normalize
Turn non-numeric columns to numeric and remove unnecessary, normalize numeric cols

✅ Keep all 77 numerical features

✅ Convert date columns into:

month
week
quarter
day of year

✅ One-hot encode:

season
crop_type
aggregation_method
market_name
report_type
etc.

✅ Drop:

window_id
raw timestamps

⚠️ Drop region_id if train and test regions are completely different (which appears to be your case).

In [ ]:
def preprocess_dataframe(df):
    df = df.copy()

    # Date columns
    date_cols = [
        'date',
        'window_start',
        'window_end',
        'release_date',
        'observation_timestamp_utc',
        'source_timestamp',
        'reporting_period_start',
        'reporting_period_end'
    ]

    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

            df[f'{col}_year'] = df[col].dt.year
            df[f'{col}_month'] = df[col].dt.month
            df[f'{col}_day'] = df[col].dt.day
            df[f'{col}_week'] = df[col].dt.isocalendar().week.astype('Int64')

    # Drop raw datetime columns
    df.drop(columns=[c for c in date_cols if c in df.columns],
            inplace=True)

    # Drop identifiers
    drop_cols = ['window_id', 'region_id']
    df.drop(columns=[c for c in drop_cols if c in df.columns],
            inplace=True)

    return df


# ----------------------------
# Apply preprocessing
# ----------------------------
train_proc = preprocess_dataframe(train_df)
valid_proc = preprocess_dataframe(valid_df)
test_proc = preprocess_dataframe(test_df)


In [ ]:
# ----------------------------
# One-hot encode together
# ----------------------------
combined = pd.concat(
    [train_proc, valid_proc, test_proc],
    keys=['train', 'valid', 'test']
)

combined = pd.get_dummies(
    combined,
    columns=[
        'season',
        'crop_type',
        'temporal_resolution',
        'aggregation_method',
        'market_name',
        'price_frequency',
        'temporal_fill_method',
        'report_type',
        'extraction_method'
    ],
    drop_first=True
)

# Split back
train_final = combined.xs('train')
valid_final = combined.xs('valid')
test_final = combined.xs('test')

# ----------------------------
# View results
# ----------------------------
print("Train Shape:", train_final.shape)
print("Valid Shape:", valid_final.shape)
print("Test Shape:", test_final.shape)

display(train_final.head())

Train Shape: (140, 118)
Valid Shape: (30, 118)
Test Shape: (30, 118)


,cloud_cover_pct,valid_pixel_count,band_B02_mean,band_B02_stddev,band_B02_p25,band_B02_p50,band_B02_p75,band_B03_mean,band_B03_stddev,band_B03_p25,...,reporting_period_end_week,season_growing,season_kharif,season_spring,season_summer,season_winter,crop_type_rice,crop_type_wheat,market_name_euronext,market_name_punjab_mandi
0,0.9780,6583,0.5008,0.0201,0.4208,0.5008,0.5808,0.3379,0.0800,0.2579,...,18,False,False,True,False,False,False,False,False,False
1,0.3140,8855,0.2499,0.0217,0.1699,0.2499,0.3299,0.5331,0.0534,0.4531,...,20,False,False,True,False,False,False,False,False,False
2,0.6617,7485,0.2418,0.0532,0.1618,0.2418,0.3218,0.4741,0.1139,0.3941,...,23,False,False,True,False,False,False,False,False,False
3,0.8128,8181,0.8105,0.1133,0.7305,0.8105,0.8905,0.1791,0.1161,0.0991,...,25,False,False,True,False,False,False,False,False,False
4,0.9100,7410,0.7417,0.0336,0.6617,0.7417,0.8217,0.6155,0.1231,0.5355,...,27,False,False,True,False,False,False,False,False,False


In [ ]:
for df in [train_final, valid_final, test_final]:
    bool_cols = df.select_dtypes(include='bool').columns
    df[bool_cols] = df[bool_cols].astype(int)

/tmp/ipykernel_1336/273833771.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[bool_cols] = df[bool_cols].astype(int)
/tmp/ipykernel_1336/273833771.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[bool_cols] = df[bool_cols].astype(int)
/tmp/ipykernel_1336/273833771.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/sta

In [ ]:
print(train_final.select_dtypes(exclude='number').columns.tolist())
print(valid_final.select_dtypes(exclude='number').columns.tolist())
print(test_final.select_dtypes(exclude='number').columns.tolist())

[]
[]
[]


In [ ]:
print(train_final.dtypes.value_counts())

float64    76
int32      24
int64      10
Int64       8
Name: count, dtype: int64


In [ ]:
train_final.shape

(140, 118)

In [ ]:
train_final.describe()

,cloud_cover_pct,valid_pixel_count,band_B02_mean,band_B02_stddev,band_B02_p25,band_B02_p50,band_B02_p75,band_B03_mean,band_B03_stddev,band_B03_p25,...,reporting_period_end_week,season_growing,season_kharif,season_spring,season_summer,season_winter,crop_type_rice,crop_type_wheat,market_name_euronext,market_name_punjab_mandi
count,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000,...,140.0,140.0,140.0,140.000000,140.000000,140.000000,140.0,140.0,140.0,140.0
mean,0.530619,7045.507143,0.516903,0.076939,0.436903,0.516903,0.596903,0.493157,0.080586,0.413157,...,27.328571,0.0,0.0,0.257143,0.257143,0.228571,0.0,0.0,0.0,0.0
std,0.291569,1718.236487,0.195167,0.040545,0.195167,0.195167,0.195167,0.209611,0.043140,0.209611,...,15.046902,0.0,0.0,0.438628,0.438628,0.421420,0.0,0.0,0.0,0.0
min,0.000900,4055.000000,0.158300,0.010300,0.078300,0.158300,0.238300,0.155100,0.010200,0.075100,...,2.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
25%,0.279525,5603.250000,0.357825,0.042325,0.277825,0.357825,0.437825,0.317350,0.044550,0.237350,...,14.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
50%,0.546650,6911.000000,0.525550,0.078050,0.445550,0.525550,0.605550,0.479600,0.075250,0.399600,...,27.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
75%,0.792075,8522.000000,0.670800,0.113725,0.590800,0.670800,0.750800,0.695150,0.119175,0.615150,...,41.0,0.0,0.0,1.000000,1.000000,0.000000,0.0,0.0,0.0,0.0
max,0.997800,9900.000000,0.849800,0.146900,0.769800,0.849800,0.929800,0.849400,0.149700,0.769400,...,53.0,0.0,0.0,1.000000,1.000000,1.000000,0.0,0.0,0.0,0.0


In [ ]:
train_final.shape, valid_final.shape, test_final.shape

((140, 118), (30, 118), (30, 118))

# Model-1) Linear Regressor

In [ ]:
train_final.columns, len(train_final.columns)

(Index(['cloud_cover_pct', 'valid_pixel_count', 'band_B02_mean',
        'band_B02_stddev', 'band_B02_p25', 'band_B02_p50', 'band_B02_p75',
        'band_B03_mean', 'band_B03_stddev', 'band_B03_p25',
        ...
        'reporting_period_end_week', 'season_growing', 'season_kharif',
        'season_spring', 'season_summer', 'season_winter', 'crop_type_rice',
        'crop_type_wheat', 'market_name_euronext', 'market_name_punjab_mandi'],
       dtype='object', length=118),
 118)

In [ ]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
target = "yield_tonnes_per_ha"

X_train = train_final.drop(columns=[target])
y_train = train_final[target]

X_valid = valid_final.drop(columns=[target])
y_valid = valid_final[target]

X_test = test_final.drop(columns=[target])
y_test = test_final[target]


In [ ]:
# Feature types
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

# Preprocessing
preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        num_cols,
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols,
    ),
])

# Model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])


In [ ]:
# Train
model.fit(X_train, y_train)

# Validation
valid_pred = model.predict(X_valid)

print("Validation RMSE:",
      np.sqrt(mean_squared_error(y_valid, valid_pred)))
print("Validation MAE:",
      mean_absolute_error(y_valid, valid_pred))
print("Validation R2:",
      r2_score(y_valid, valid_pred))


Validation RMSE: 0.02841421341944479
Validation MAE: 0.023562814144560067
Validation R2: 0.9312515531541653


In [ ]:
# Save checkpoint
joblib.dump(model, "best_linear_regressor.pkl")

# Load checkpoint
model = joblib.load("best_linear_regressor.pkl")

# Test
test_pred = model.predict(X_test)

print("Test RMSE:",
      np.sqrt(mean_squared_error(y_test, test_pred)))
print("Test MAE:",
      mean_absolute_error(y_test, test_pred))
print("Test R2:",
      r2_score(y_test, test_pred))

Test RMSE: 0.03142443378606909
Test MAE: 0.02404263564133658
Test R2: 0.9398924566393899


# Model-2) LightGBM

In [ ]:
from lightgbm import LGBMRegressor

In [ ]:
TARGET = "yield_tonnes_per_ha"

# Split features and target
X_train = train_final.drop(columns=[TARGET])
y_train = train_final[TARGET]

X_valid = valid_final.drop(columns=[TARGET])
y_valid = valid_final[TARGET]

X_test = test_final.drop(columns=[TARGET])
y_test = test_final[TARGET]

In [ ]:
# LightGBM model
model = LGBMRegressor(
    objective="regression",
    n_estimators=5000,
    learning_rate=0.01,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=-1
)

# Train with early stopping
model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="rmse",
    callbacks=[
        __import__("lightgbm").early_stopping(200),
        __import__("lightgbm").log_evaluation(100)
    ]
)

Training until validation scores don't improve for 200 rounds
[100]	valid_0's rmse: 0.0552334	valid_0's l2: 0.00305073
[200]	valid_0's rmse: 0.0405593	valid_0's l2: 0.00164506
[300]	valid_0's rmse: 0.035027	valid_0's l2: 0.00122689
[400]	valid_0's rmse: 0.0341572	valid_0's l2: 0.00116672
[500]	valid_0's rmse: 0.0339754	valid_0's l2: 0.00115433
[600]	valid_0's rmse: 0.033838	valid_0's l2: 0.00114501
[700]	valid_0's rmse: 0.0337336	valid_0's l2: 0.00113796
[800]	valid_0's rmse: 0.0335028	valid_0's l2: 0.00112244
[900]	valid_0's rmse: 0.0332877	valid_0's l2: 0.00110807
[1000]	valid_0's rmse: 0.0332495	valid_0's l2: 0.00110553
[1100]	valid_0's rmse: 0.0333526	valid_0's l2: 0.00111239
Early stopping, best iteration is:
[962]	valid_0's rmse: 0.0331656	valid_0's l2: 0.00109996


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.01, n_estimators=5000,
              objective='regression', random_state=42, subsample=0.8,
              verbosity=-1)

In [ ]:
# Save best checkpoint
joblib.dump(model, "best_lightgbm.pkl")

# Load best checkpoint
model = joblib.load("best_lightgbm.pkl")

# Validation predictions
valid_pred = model.predict(X_valid)

valid_rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
valid_mae = mean_absolute_error(y_valid, valid_pred)
valid_r2 = r2_score(y_valid, valid_pred)

print("\nVALIDATION")
print(f"RMSE : {valid_rmse:.4f}")
print(f"MAE  : {valid_mae:.4f}")
print(f"R²   : {valid_r2:.4f}")


VALIDATION
RMSE : 0.0332
MAE  : 0.0269
R²   : 0.9063


In [ ]:
# Test predictions
test_pred = model.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
test_mae = mean_absolute_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)

print("\nTEST")
print(f"RMSE : {test_rmse:.4f}")
print(f"MAE  : {test_mae:.4f}")
print(f"R²   : {test_r2:.4f}")



TEST
RMSE : 0.0375
MAE  : 0.0303
R²   : 0.9145


In [ ]:
# Feature importance
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 20 Features")
print(importance.head(20))


Top 20 Features
                    feature  importance
69              rainfall_mm         582
72  shortwave_radiation_sum         542
47                 evi_mean         281
37                ndvi_mean         151
39                 ndvi_min         145
27            band_B8A_mean         130
68        temperature_min_c         126
13          band_B04_stddev         121
33          band_B11_stddev         118
75        production_tonnes         108
28          band_B8A_stddev          97
59                 savi_min          96
0           cloud_cover_pct          92
57                savi_mean          87
32            band_B11_mean          83
18          band_B05_stddev          71
79                date_week          71
38              ndvi_stddev          68
8           band_B03_stddev          66
12            band_B04_mean          66


# Model-3) LSTM

In [ ]:
from sklearn.preprocessing import StandardScaler
from torchmetrics import MeanAbsoluteError
from torchmetrics import R2Score
from torchmetrics import MeanSquaredError

In [ ]:
TARGET = "yield_tonnes_per_ha"

X_train = train_final.drop(columns=[TARGET]).values
y_train = train_final[TARGET].values

X_valid = valid_final.drop(columns=[TARGET]).values
y_valid = valid_final[TARGET].values

X_test = test_final.drop(columns=[TARGET]).values
y_test = test_final[TARGET].values

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

joblib.dump(scaler, "lstm_scaler.pkl")

# LSTM input format:
# (N, seq_len, input_size)

X_train = X_train.reshape(len(X_train), X_train.shape[1], 1)
X_valid = X_valid.reshape(len(X_valid), X_valid.shape[1], 1)
X_test = X_test.reshape(len(X_test), X_test.shape[1], 1)

In [ ]:
class YieldDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = YieldDataset(X_train, y_train)
valid_ds = YieldDataset(X_valid, y_valid)
test_ds = YieldDataset(X_test, y_test)

train_loader = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_ds,
    batch_size=16,
    shuffle=False
)

In [ ]:
from torchmetrics import MeanAbsoluteError, R2Score, MeanSquaredError

class LSTMRegressor(L.LightningModule):

    def __init__(
        self,
        input_size=1,
        hidden_size=32,
        num_layers=1,
        lr=1e-3
    ):
        super().__init__()

        self.save_hyperparameters()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.head = nn.Linear(hidden_size, 1)

        self.loss_fn = nn.MSELoss()

        # TorchMetrics
        self.test_mae = MeanAbsoluteError()
        self.test_r2 = R2Score()
        self.test_rmse = MeanSquaredError(squared=False)

    def forward(self, x):

        _, (hidden, _) = self.lstm(x)

        return self.head(hidden[-1]).squeeze(1)

    def training_step(self, batch, batch_idx):

        x, y = batch

        pred = self(x)

        loss = self.loss_fn(pred, y)

        self.log(
            "train_loss",
            loss,
            prog_bar=True,
            on_epoch=True
        )

        return loss

    def validation_step(self, batch, batch_idx):

        x, y = batch

        pred = self(x)

        loss = self.loss_fn(pred, y)

        self.log(
            "val_loss",
            loss,
            prog_bar=True,
            on_epoch=True
        )

        return loss

    def test_step(self, batch, batch_idx):

        x, y = batch

        pred = self(x)

        loss = self.loss_fn(pred, y)

        self.test_rmse.update(pred, y)
        self.test_mae.update(pred, y)
        self.test_r2.update(pred, y)

        self.log(
            "test_loss",
            loss,
            prog_bar=True,
            on_epoch=True
        )

        return loss

    def on_test_epoch_end(self):

        rmse = self.test_rmse.compute().item()
        mae = self.test_mae.compute().item()
        r2 = self.test_r2.compute().item()

        print("\n===== TEST METRICS =====")
        print(f"RMSE : {rmse:.4f}")
        print(f"MAE  : {mae:.4f}")
        print(f"R2   : {r2:.4f}")

        self.log("test_rmse", rmse)
        self.log("test_mae", mae)
        self.log("test_r2", r2)

        self.test_rmse.reset()
        self.test_mae.reset()
        self.test_r2.reset()

    def configure_optimizers(self):

        return torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr
        )

In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints",
    filename="best-lstm",
    monitor="val_loss",
    mode="min",
    save_top_k=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=20,
    mode="min"
)

In [ ]:
model = LSTMRegressor(
    hidden_size=64,
    num_layers=2,
    lr=1e-3
)

trainer = L.Trainer(
    max_epochs=200,
    enable_progress_bar=False,
    enable_model_summary=False,
    callbacks=[
        checkpoint_callback,
        early_stop
    ]
)

trainer.fit(
    model,
    train_loader,
    valid_loader
)

print("Best model:")
print(checkpoint_callback.best_model_path)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:321: The number of training batches (9) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Best model:
/content/checkpoints/best-lstm-v4.ckpt


In [ ]:
best_model = LSTMRegressor.load_from_checkpoint(
    checkpoint_callback.best_model_path
)

best_model.eval()

LSTMRegressor(
  (lstm): LSTM(1, 64, num_layers=2, batch_first=True)
  (head): Linear(in_features=64, out_features=1, bias=True)
  (loss_fn): MSELoss()
  (test_mae): MeanAbsoluteError()
  (test_r2): R2Score()
  (test_rmse): MeanSquaredError()
)

In [ ]:
trainer.test(
    model=best_model,
    dataloaders=test_loader
)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



===== TEST METRICS =====
RMSE : 0.1321
MAE  : 0.1083
R2   : -0.0617


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    0.01744232513010502    │
│         test_mae          │    0.10828341543674469    │
│          test_r2          │   -0.06169271469116211    │
│         test_rmse         │     0.132069393992424     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.01744232513010502,
  'test_rmse': 0.132069393992424,
  'test_mae': 0.10828341543674469,
  'test_r2': -0.06169271469116211}]